# 02 · Station map

Where are our stations, and where are trains late most often?

Steps:
1. Load `config/stations.csv` (built by `db_delay_mlops.data.stations`)
2. Compute the late share (arrival ≥ 6 min) per EVA number for one month
3. Merge the EVA numbers of one station (e.g. Berlin Hbf has three) into one point
4. Draw an interactive map

In [1]:
from pathlib import Path

import plotly.express as px
import polars as pl

DATA_DIR = Path("../data/raw/hf/monthly_processed_data")
MONTH = "2026-09"  # change me: any downloaded month

## 1. Stations

The 12 bus stations have no StaDa entry, so they have no coordinates and cannot be drawn.

In [2]:
stations = pl.read_csv("../config/stations.csv", schema_overrides={"eva": pl.String})
print(stations.height, "EVA numbers,", stations["collect"].sum(), "with StaDa data")
stations.group_by("federal_state").len().sort("len", descending=True)

146 EVA numbers, 134 with StaDa data


federal_state,len
str,u32
"""Nordrhein-Westfalen""",25
"""Berlin""",23
"""Bayern""",16
"""Baden-Württemberg""",15
null,12
…,…
"""Sachsen-Anhalt""",2
"""Thüringen""",2
"""Saarland""",1


## 2. Late share per EVA number

Same label as in notebook 01: arrival delay computed from the arrival columns, cancelled stops and first stops of a ride (no arrival) removed.

In [3]:
late_per_eva = (
    # Lazily scan the monthly Parquet file (nothing is read yet, only a query plan is built)
    pl.scan_parquet(DATA_DIR / f"data-{MONTH}.parquet")
    # Keep only the 4 columns we need (projection pushdown: other columns are never read from disk)
    .select("eva", "arrival_planned_time", "arrival_change_time", "arrival_is_canceled")
    .filter(
        # Only stations from our `stations` DataFrame; implode() turns the column into a single list value
        pl.col("eva").is_in(stations["eva"].implode()),
        # Only rows that actually have a planned arrival (drops departures-only rows, e.g. at origin stations)
        pl.col("arrival_planned_time").is_not_null(),
        # Exclude canceled arrivals (~ is logical NOT)
        ~pl.col("arrival_is_canceled"),
    )
    # Delay in whole minutes = actual/changed arrival minus planned arrival
    .with_columns(delay=(pl.col("arrival_change_time") - pl.col("arrival_planned_time")).dt.total_minutes())
    # One group per station
    .group_by("eva")
    # stops = number of arrivals; late_stops = number of arrivals with delay >= 6 minutes
    .agg(stops=pl.len(), late_stops=(pl.col("delay") >= 6).sum())
    # Execute the whole query plan and return an eager DataFrame
    .collect()
)
# Show the 5 busiest stations (most arrivals)
late_per_eva.sort("stops", descending=True).head()

eva,stops,late_stops
str,u32,u32
"""08089028""",46892,1590
"""08089066""",42490,1462
"""08098549""",35752,3190
"""08089015""",33670,1143
"""08000207""",32398,10696


## 3. One point per station

Group by `stada_name`. We **sum** stops and late stops and divide afterwards. Averaging the late shares of the EVA numbers would give a small S-Bahn platform the same weight as the main hall.

Coordinates: those of the main EVA number (`is_main_eva`), otherwise the first one.

In [6]:
per_station = (
    stations.filter(pl.col("collect"))
    .join(late_per_eva, on="eva", how="left")
    .sort("is_main_eva", descending=True)  # main EVA first, so .first() picks its coordinates
    .group_by("stada_name")
    .agg(
        pl.col("federal_state").first(),
        pl.col("category").first(),
        pl.col("lat").first(),
        pl.col("lon").first(),
        pl.len().alias("n_evas"),
        pl.col("stops").sum(),
        pl.col("late_stops").sum(),
    )
    .with_columns(late_share=(pl.col("late_stops") / pl.col("stops")).round(3))
    .sort("late_share", descending=True)
)
print(per_station.height, "stations")
per_station.head(10)

110 stations


stada_name,federal_state,category,lat,lon,n_evas,stops,late_stops,late_share
str,str,i64,f64,f64,u32,u32,u32,f64
"""Hamm (Westf) Hbf""","""Nordrhein-Westfalen""",2,51.678078,7.807821,1,9426,5341,0.567
"""Oberhausen Hbf""","""Nordrhein-Westfalen""",2,51.474878,6.851588,1,10703,5620,0.525
"""Fulda""","""Hessen""",2,50.554906,9.684169,1,6456,3219,0.499
"""Bielefeld Hbf""","""Nordrhein-Westfalen""",2,52.029261,8.532722,1,8861,4396,0.496
"""Gelsenkirchen Hbf""","""Nordrhein-Westfalen""",2,51.504903,7.102455,1,12481,6096,0.488
"""Wolfsburg Hbf""","""Niedersachsen""",2,52.429498,10.787784,1,4471,2143,0.479
"""Hannover Hbf""","""Niedersachsen""",1,52.376761,9.741021,1,19663,8880,0.452
"""Koblenz Hbf""","""Rheinland-Pfalz""",2,50.350929,7.588345,1,5457,2403,0.44
"""Tübingen Hbf""","""Baden-Württemberg""",2,48.515641,9.055395,1,4491,1954,0.435


## 4. Map

Point size = number of arrivals, colour = late share. Hover over a point for details.

In [7]:
fig = px.scatter_map(
    per_station,
    lat="lat",
    lon="lon",
    color="late_share",
    size="stops",
    hover_name="stada_name",
    hover_data={"federal_state": True, "stops": True, "late_share": ":.1%", "lat": False, "lon": False},
    color_continuous_scale="RdYlGn_r",  # green = on time, red = late
    size_max=25,
    zoom=4.8,
    center={"lat": 51.2, "lon": 10.4},
    map_style="carto-positron",
    title=f"Share of arrivals ≥ 6 min late, {MONTH}",
    height=750,
)
fig.update_layout(coloraxis_colorbar_tickformat=".0%")
fig.show()

## Questions

1. Do you see a regional pattern (north/south, east/west)?
2. Are hubs with many stops (big points) later than small stations?
3. Change `MONTH` to a winter month (e.g. `2025-01`) and to `2024-07`. What changes?
4. Could "region" or "station size" be a useful feature? Which leakage rule would it have to follow?